# Ch 25 · Profiling, Benchmarking and Tuning for Throughput

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/25-profiling-benchmarking-tuning.ipynb)

Companion notebook for the chapter page `chapters/25-profiling-benchmarking-tuning.html` ([page](../chapters/25-profiling-benchmarking-tuning.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

This notebook mirrors the chapter page: every example in order. On a GPU runtime the examples call the real `mjwarp-testspeed` and `EventTracer`; on CPU they print a skip note and run a small hand-made stand-in. Numbers printed on CPU are CPU numbers, not GPU performance; on Colab you get your own GPU numbers.

## 1. Profile one captured step with `EventTracer`
`with EventTracer(enabled=ON_GPU)` around a captured step; `trace()` after each replay gives `{name: (ms, sub)}`. Divide by `nworld` (and the number of replays) for ns per world-step, the unit testspeed prints.

In [ ]:
# Example 25.1 - ex1_event_trace.py
"""Example 25.1 - profile one captured mjw.step with EventTracer; read ns per world-step.

GPU: `with EventTracer(enabled=ON_GPU) as t:` around a captured step, replay it NSTEP times,
then t.trace() gives {name: (times_ms, sub_trace)} for every @event_scope function. We print
the tree in microseconds per replay and in nanoseconds per world-step (testspeed's unit).
CPU: EventTracer is enabled=False (Rule G: the first event_scope would raise "Device cpu is not
a CUDA device"), so the trace is empty. As a CPU stand-in we record the SAME tree by watching
the event_scope wrapper with sys.setprofile (skipping capture_while/capture_if bodies, as
event_scope does) and timing it with the wall clock. Those numbers are CPU numbers with Python
overhead included, never GPU performance.
Run: python ex1_event_trace.py [--dump-json PATH]
  --dump-json PATH writes the nested trace (ns per world-step) for the page's icicle chart.
"""
import argparse
import json
import sys
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath
from mujoco_warp._src import warp_util          # private API: EventTracer, event_scope

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 20

HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
mjm = mujoco.MjModel.from_xml_path(str(HUMANOID_PATH))
mjd = mujoco.MjData(mjm)
mujoco.mj_resetDataKeyframe(mjm, mjd, 0)                # like testspeed's --keyframe 0
m = mjw.put_model(mjm)
if ON_GPU and not wp.is_conditional_graph_supported():
    m.opt.graph_conditional = False                     # Rule G fallback (CUDA < 12.4)
d = mjw.put_data(mjm, mjd, nworld=NWORLD, nconmax=24, njmax=64)
mjw.step(m, d)                                          # warm-up: JIT compile outside the trace
wp.synchronize()


def sum_trace(a, b):
    """Add two traces call by call (what testspeed._sum_trace does after every replay)."""
    out = dict(a)
    for name, (times, sub) in b.items():
        if name in a:
            times = tuple(t1 + t2 for t1, t2 in zip(a[name][0], times))
            sub = sum_trace(a[name][1], sub)
        out[name] = (times, sub)
    return out


def per_world_step(trace, nstep, nworld, reps=1):
    """{name: (ms, ...)} -> {name: {"us": per replay, "ns": per world-step, "calls", "sub"}}.

    us per replay = 1e3 * ms / nstep (the latency of one call for all worlds);
    ns per world-step = 1e6 * ms / (nstep * nworld), the unit testspeed prints."""
    out = {}
    for name, (times, sub) in trace.items():
        out[name] = {"us": round(1e3 * sum(times) / nstep, 3),
                     "ns": round(1e6 * sum(times) / (nstep * nworld), 3),
                     "calls": len(times) // reps,
                     "sub": per_world_step(sub, nstep, nworld, reps)}
    return out


def show(tree, indent=0, depth=4):
    if indent == 0:
        print(f"{'':<28} {'us/replay':>12} {'ns/world-step':>14}")
    for name, node in tree.items():
        print("  " * indent + f"{name:<{28 - 2 * indent}} {node['us']:>12.1f} {node['ns']:>14.1f}")
        if indent + 1 < depth:
            show(node["sub"], indent + 1, depth)


# ---- 1. the real tool: EventTracer (records CUDA events, so GPU only) ----
with warp_util.EventTracer(enabled=ON_GPU) as tracer:
    with wp.ScopedCapture() as cap:                     # events become graph nodes
        mjw.step(m, d)
    gpu_trace = {}
    for _ in range(NSTEP):
        wp.capture_launch(cap.graph)
        wp.synchronize()
        # trace() reads the events of the replay that just ran; add them up
        gpu_trace = sum_trace(gpu_trace, tracer.trace())

if ON_GPU:
    tree = per_world_step(gpu_trace, NSTEP, NWORLD)
    print(f"EventTracer on {wp.get_device()}, nworld={NWORLD}, {NSTEP} replays:")
    show(tree)
    source = "EventTracer, GPU"
else:
    print("skipped on CPU: EventTracer needs CUDA events; trace() returned", gpu_trace)

    # ---- 2. CPU stand-in: the same tree, timed with the wall clock ----
    WRAPPER = warp_util.event_scope(lambda: None).__code__   # every event_scope shares it
    root = {}
    stack, open_ = [root], []

    inside_conditional = [0]                            # event_scope skips these bodies

    def watch(frame, event, arg):
        code = frame.f_code
        if code.co_name in ("capture_while", "capture_if") and event in ("call", "return"):
            inside_conditional[0] += 1 if event == "call" else -1
            return
        if code is not WRAPPER or inside_conditional[0]:
            return
        if event == "call":                             # the wrapper's closure holds `name`
            node = stack[-1].setdefault(frame.f_locals["name"], ([], {}))
            stack.append(node[1])
            open_.append((node, time.perf_counter()))
        elif event == "return":
            node, t0 = open_.pop()
            stack.pop()
            node[0].append(1e3 * (time.perf_counter() - t0))   # ms, like get_event_elapsed_time

    sys.setprofile(watch)
    for _ in range(NSTEP):
        mjw.step(m, d)                                  # eager: CPU launches run immediately
    sys.setprofile(None)
    tree = per_world_step(root, NSTEP, NWORLD, reps=NSTEP)
    print(f"CPU stand-in (wall clock, eager, nworld={NWORLD}, {NSTEP} steps, CPU numbers):")
    show(tree)
    source = "sys.setprofile wall clock, CPU"

step_ns = tree["step"]["ns"]
fwd = tree["step"]["sub"]["forward"]["sub"]
top = max(fwd, key=lambda k: fwd[k]["ns"])
print(f"step = {step_ns:.0f} ns per world-step -> {1e9 / step_ns:,.0f} world-steps/s")
print(f"largest stage under forward: {top} ({100 * fwd[top]['ns'] / step_ns:.0f}% of step)")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"source": source, "nworld": NWORLD, "nstep": NSTEP, "unit": "ns/world-step",
                   "tree": tree}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

Draw Figure 25.5 from the trace you just recorded: on a GPU runtime this is your own `EventTracer` tree; on CPU it is the wall-clock stand-in, labelled as such.

In [ ]:
# Figure 25.5 from YOUR run: an icicle chart of `tree` (left by Example 25.1).
# Width = ns per world-step, rows = nesting depth. Numbers above were printed first (Rule O).
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("skipped: matplotlib not installed")
else:
    label = f"your GPU ({wp.get_device()})" if ON_GPU else "CPU stand-in (wall clock), not GPU performance"
    fig, ax = plt.subplots(figsize=(11, 3.2))
    colors = plt.cm.tab20.colors

    def draw(nodes, x0, depth, scale, max_depth=4):
        x = x0
        for i, (name, node) in enumerate(nodes.items()):
            w = node["ns"] * scale
            ax.barh(-depth, w, left=x, height=0.9, color=colors[(depth * 5 + i) % 20],
                    edgecolor="white", linewidth=0.5)
            if w > 0.04:                              # label only boxes wide enough to read
                ax.text(x + w / 2, -depth, name, ha="center", va="center", fontsize=7, clip_on=True)
            if depth + 1 < max_depth:
                draw(node["sub"], x, depth + 1, scale, max_depth)
            x += w

    draw(tree, 0.0, 0, 1.0 / tree["step"]["ns"])
    ax.set_xlim(0, 1); ax.set_yticks([0, -1, -2, -3], ["step", "depth 1", "depth 2", "depth 3"])
    ax.set_xlabel(f"fraction of step = {tree['step']['ns']:,.0f} ns per world-step")
    ax.set_title(f"EventTracer tree, {label}")
    plt.tight_layout(); plt.show()

## 2. `mjwarp-testspeed` and its JSON
The model path falls back to the installed package, so `test_data/humanoid/humanoid.xml` works without a clone. On CPU testspeed raises `ValueError: testspeed available for gpu only`; the example then runs `mini_testspeed`.

In [ ]:
# Example 25.2 - ex2_testspeed.py
"""Example 25.2 - run mjwarp-testspeed on the shipped humanoid and parse its JSON.

GPU: runs `mjwarp-testspeed test_data/humanoid/humanoid.xml --nworld 4096 --nstep 500
--format json --event_trace` in a subprocess (the path falls back to the installed package, so
no clone is needed) and prints steps/s, ncon/nefc mean and p95, solver iterations, and the
step.forward.* stages (the trace keys exist only with --event_trace).
CPU: testspeed refuses to run (Rule G). We show its real error, then run `mini_testspeed`, a
small re-implementation of what testspeed measures (testspeed.py::_main + cli.unroll), and
print the same JSON keys except total_memory (a device free-memory difference). CPU numbers
are CPU numbers, never GPU performance.
Run: python ex2_testspeed.py [--dump-json PATH]
  --dump-json PATH writes the CPU metrics in testspeed's JSON schema (bundled sample).
"""
import argparse
import dataclasses
import json
import subprocess
import sys
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 500 if ON_GPU else 50
MODEL = "test_data/humanoid/humanoid.xml"


def run_testspeed(*flags):
    """Run mjwarp-testspeed in a subprocess; return (returncode, stdout, last stderr line)."""
    cmd = [sys.executable, "-m", "mujoco_warp.testspeed", MODEL, *flags]
    r = subprocess.run(cmd, capture_output=True, text=True)
    err = [ln for ln in r.stderr.splitlines() if ln.strip()]
    return r.returncode, r.stdout, err[-1] if err else ""


# --- a copy of cli._ctrl_noise (Ornstein-Uhlenbeck noise on ctrl, Halton samples) ---
@wp.func
def halton(index: int, base: int) -> float:
    n0 = index
    b = float(base)
    f = float(1.0) / b
    hn = float(0.0)
    while n0 > 0:
        n1 = n0 // base
        r = n0 - n1 * base
        hn += f * float(r)
        f /= b
        n0 = n1
    return hn


@wp.kernel
def ctrl_noise(timestep: wp.array(dtype=float), ctrllimited: wp.array(dtype=bool),
               ctrlrange: wp.array2d(dtype=wp.vec2), ctrl_in: wp.array2d(dtype=float),
               center: wp.array(dtype=float), step: int, std: float, rate_s: float,
               ctrl_out: wp.array2d(dtype=float)):
    worldid, actid = wp.tid()
    rate = wp.exp(-timestep[worldid % timestep.shape[0]] / rate_s)
    scale = std * wp.sqrt(1.0 - rate * rate)
    midpoint = 0.0
    halfrange = 1.0
    cr = ctrlrange[worldid % ctrlrange.shape[0], actid]
    if ctrllimited[actid]:
        midpoint = 0.5 * (cr[1] + cr[0])
        halfrange = 0.5 * (cr[1] - cr[0])
    if center.shape[0] > 0:
        midpoint = center[actid]
    ctrl = rate * ctrl_in[worldid, actid] + (1.0 - rate) * midpoint      # pull toward center
    ctrl += scale * halfrange * (2.0 * halton((step + 1) * (worldid + 1), actid + 2) - 1.0)
    if ctrllimited[actid]:
        ctrl = wp.clamp(ctrl, cr[0], cr[1])
    ctrl_out[worldid, actid] = ctrl


def nbytes(obj):
    """Sum of wp.array capacities in a dataclass (testspeed._dataclass_memory)."""
    total = 0
    for f in dataclasses.fields(obj):
        v = getattr(obj, f.name)
        if dataclasses.is_dataclass(v):
            total += nbytes(v)
        elif isinstance(v, wp.array):
            total += v.capacity
    return total


def mini_testspeed(mjm, fn=mjw.step, nworld=8, nstep=50, noise_std=0.01, noise_rate=0.1,
                   keyframe=0, nconmax=None, njmax=None):
    """What testspeed measures: capture fn once, then per step noise -> replay -> read counts."""
    mjd = mujoco.MjData(mjm)
    ctrls = None
    if mjm.nkey > 0 and keyframe > -1:              # cli.init_structs: keyframe -> ctrl center
        mujoco.mj_resetDataKeyframe(mjm, mjd, keyframe)
        ctrls = [mjd.ctrl.copy() for _ in range(nstep)]
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0                         # testspeed's default overflow_behavior=error
    d = mjw.put_data(mjm, mjd, nworld=nworld, nconmax=nconmax, njmax=njmax)
    t0 = time.perf_counter()
    with wp.ScopedCapture() as cap:                 # JIT + capture: testspeed's "jit_duration"
        fn(m, d)
    jit = time.perf_counter() - t0
    nacon, nefc, niter, runtime = [], [], [], 0.0
    for i in range(nstep):
        if ctrls is not None:                       # no keyframe and no --replay: no noise!
            center = wp.array(ctrls[i], dtype=float)
            wp.launch(ctrl_noise, dim=(d.nworld, m.nu),
                      inputs=[m.opt.timestep, m.actuator_ctrllimited, m.actuator_ctrlrange,
                              d.ctrl, center, i, noise_std, noise_rate], outputs=[d.ctrl])
            wp.synchronize()
        t = time.perf_counter()
        wp.capture_launch(cap.graph)                # only the replay is timed
        wp.synchronize()
        runtime += time.perf_counter() - t
        nacon.append(max(d.nacon.numpy()[0], d.ncollision.numpy()[0]))   # total, all worlds
        nefc.append(d.nefc.numpy().max())                                # worst world
        niter.append(d.solver_niter.numpy())
        if d.overflow.numpy().any():
            raise RuntimeError(f"overflow {d.overflow.numpy()} - raise nconmax/njmax")
    steps = nworld * nstep
    qpos = d.qpos.numpy()
    return {
        "jit_duration": jit, "run_time": runtime, "steps_per_second": steps / runtime,
        "converged_worlds": int(np.sum(~np.any(np.isnan(qpos), axis=1))),
        "model_memory": nbytes(m), "data_memory": nbytes(d),
        "ncon_mean": float(np.mean(nacon) / nworld),
        "ncon_p95": float(np.percentile(nacon, 95) / nworld),
        "nefc_mean": float(np.mean(nefc)), "nefc_p95": float(np.percentile(nefc, 95)),
        "solver_niter_mean": float(np.mean(niter)),
        "solver_niter_p95": float(np.percentile(niter, 95)),
    }, d


def summary(metrics):
    print(f"  steps/s           {metrics['steps_per_second']:,.0f}")
    print(f"  ncon  mean / p95  {metrics['ncon_mean']:.2f} / {metrics['ncon_p95']:.2f} per world")
    print(f"  nefc  mean / p95  {metrics['nefc_mean']:.1f} / {metrics['nefc_p95']:.1f}"
          " (worst world)")
    print(f"  solver_niter mean {metrics['solver_niter_mean']:.2f},"
          f" p95 {metrics['solver_niter_p95']:.0f}")
    print(f"  converged worlds  {metrics['converged_worlds']}")


if __name__ == "__main__":
    if ON_GPU:
        code, out, err = run_testspeed("--nworld", str(NWORLD), "--nstep", str(NSTEP),
                                       "--format", "json", "--event_trace")   # trace keys too
        if code != 0:
            raise RuntimeError(f"testspeed failed: {err}")
        metrics = json.loads(out.strip().splitlines()[-1])
        print(f"mjwarp-testspeed {MODEL} nworld={NWORLD} nstep={NSTEP} (your GPU):")
        summary(metrics)
        stages = {k: v for k, v in metrics.items()     # direct children of step.forward only
                  if k.startswith("step.forward.") and k.count(".") == 2}
        print("  stages [ns/world-step]:", {k[13:]: round(v, 1) for k, v in stages.items()})
    else:
        code, _, err = run_testspeed("--nworld", "8", "--nstep", "5", "--format", "json")
        print(f"skipped on CPU: mjwarp-testspeed exited with code {code}: {err}")
        mjm = mujoco.MjModel.from_xml_path(str(epath.resource_path("mujoco_warp") / MODEL))
        metrics, d = mini_testspeed(mjm, nworld=NWORLD, nstep=NSTEP, nconmax=24, njmax=64)
        print(f"mini_testspeed (CPU stand-in) {MODEL} nworld={NWORLD} nstep={NSTEP}:")
        summary(metrics)
        print("  JSON keys:", ", ".join(metrics))
    if args.dump_json:
        with open(args.dump_json, "w") as f:
            json.dump({k: (round(v, 4) if isinstance(v, float) else v)
                       for k, v in metrics.items()}, f)
        print("wrote", args.dump_json)

## 3. What `--noise_std` and `--function` change
Without ctrl noise every world repeats world 0; `--function forward` never advances time. `--function` accepts only public functions whose parameters are exactly `(m, d)` or `(m, d, rc)`.

In [ ]:
# Example 25.3 - ex3_testspeed_noise_function.py
"""Example 25.3 - what --noise_std and --function change in a testspeed measurement.

GPU: four testspeed runs on the humanoid: the default noise (0.01) vs --noise_std 0,
--function forward vs --function step, and --function render at 64x64 with depth; prints a
comparison table (steps/s, ncon).
CPU: testspeed does not run (Rule G). We print the list of names --function accepts (built with
testspeed's own rule), then measure the two effects that matter for comparisons with a small
copy of testspeed's loop: (1) without noise every world stays identical, (2) `forward` never
advances time, so it times the same state again and again. CPU numbers are CPU numbers.
Run: python ex3_testspeed_noise_function.py
"""
import inspect
import subprocess
import sys
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 500 if ON_GPU else 50
MODEL = "test_data/humanoid/humanoid.xml"

# testspeed.py::_FUNCS: public functions whose parameters are exactly {m, d} or {m, d, rc}
FUNCS = sorted(n for n, f in inspect.getmembers(mjw, inspect.isfunction)
               if set(inspect.signature(f).parameters) in ({"m", "d"}, {"m", "d", "rc"}))
print(f"--function accepts {len(FUNCS)} names:", ", ".join(FUNCS))
for name in ("fwd_position", "collision", "fwd_acceleration"):
    params = list(inspect.signature(getattr(mjw, name)).parameters)
    print(f"  not accepted: {name}{tuple(params)} (extra parameter)")

# --- a copy of cli._ctrl_noise (Ornstein-Uhlenbeck noise on ctrl, Halton samples) ---
@wp.func
def halton(index: int, base: int) -> float:
    n0 = index
    b = float(base)
    f = float(1.0) / b
    hn = float(0.0)
    while n0 > 0:
        n1 = n0 // base
        r = n0 - n1 * base
        hn += f * float(r)
        f /= b
        n0 = n1
    return hn


@wp.kernel
def ctrl_noise(timestep: wp.array(dtype=float), ctrllimited: wp.array(dtype=bool),
               ctrlrange: wp.array2d(dtype=wp.vec2), ctrl_in: wp.array2d(dtype=float),
               center: wp.array(dtype=float), step: int, std: float, rate_s: float,
               ctrl_out: wp.array2d(dtype=float)):
    worldid, actid = wp.tid()
    rate = wp.exp(-timestep[worldid % timestep.shape[0]] / rate_s)
    scale = std * wp.sqrt(1.0 - rate * rate)
    midpoint = 0.0
    halfrange = 1.0
    cr = ctrlrange[worldid % ctrlrange.shape[0], actid]
    if ctrllimited[actid]:
        midpoint = 0.5 * (cr[1] + cr[0])
        halfrange = 0.5 * (cr[1] - cr[0])
    if center.shape[0] > 0:
        midpoint = center[actid]
    ctrl = rate * ctrl_in[worldid, actid] + (1.0 - rate) * midpoint      # pull toward center
    ctrl += scale * halfrange * (2.0 * halton((step + 1) * (worldid + 1), actid + 2) - 1.0)
    if ctrllimited[actid]:
        ctrl = wp.clamp(ctrl, cr[0], cr[1])
    ctrl_out[worldid, actid] = ctrl


mjm = mujoco.MjModel.from_xml_path(str(epath.resource_path("mujoco_warp") / MODEL))
mjd = mujoco.MjData(mjm)
mujoco.mj_resetDataKeyframe(mjm, mjd, 0)
center = wp.array(mjd.ctrl.astype(np.float32), dtype=float)


def run(fn, noise_std):
    m = mjw.put_model(mjm)
    d = mjw.put_data(mjm, mjd, nworld=NWORLD, nconmax=24, njmax=64)
    with wp.ScopedCapture() as cap:
        fn(m, d)
    ncon, runtime = [], 0.0
    for i in range(NSTEP):
        wp.launch(ctrl_noise, dim=(NWORLD, m.nu),
                  inputs=[m.opt.timestep, m.actuator_ctrllimited, m.actuator_ctrlrange, d.ctrl,
                          center, i, noise_std, 0.1], outputs=[d.ctrl])
        t = time.perf_counter()
        wp.capture_launch(cap.graph)
        wp.synchronize()
        runtime += time.perf_counter() - t
        # testspeed's ncon_mean: max(nacon, ncollision) / nworld (testspeed.py:263)
        ncon.append(max(d.nacon.numpy()[0], d.ncollision.numpy()[0]) / NWORLD)
    q = d.qpos.numpy()
    return {"sps": NWORLD * NSTEP / runtime, "ncon": np.mean(ncon),
            "spread": float(np.abs(q - q[0]).max()), "time": float(d.time.numpy()[0])}



if ON_GPU:
    import json

    def testspeed(*flags):
        cmd = [sys.executable, "-m", "mujoco_warp.testspeed", MODEL, "--nworld", str(NWORLD),
               "--nstep", str(NSTEP), "--format", "json", *flags]
        out = subprocess.run(cmd, capture_output=True, text=True, check=True).stdout
        return json.loads(out.strip().splitlines()[-1])

    runs = {
        "step, noise 0.01 (default)": testspeed(),
        "step, --noise_std 0": testspeed("--noise_std", "0"),
        "--function forward": testspeed("--function", "forward"),
        "--function render 64x64 rgb+depth": testspeed(
            "--function", "render", "--render_width", "64", "--render_height", "64",
            "--render_depth", "true"),
    }
    print(f"\n{'run':<36}{'steps/s':>14}{'ncon mean':>11}")
    for name, r in runs.items():
        print(f"{name:<36}{r['steps_per_second']:>14,.0f}{r['ncon_mean']:>11.2f}")
else:
    print("\nskipped on CPU: mjwarp-testspeed is GPU only. What the four GPU runs measure:")
    print("  default            : step; ctrl pulled to the keyframe ctrl + OU noise (0.01, 0.1 s)")
    print("  --noise_std 0      : step, no noise -> every world repeats world 0")
    print("  --function forward : forward only, no integration, d.time never moves")
    print("  --function render  : refit_bvh + render timed; step replayed untimed in between")
    print("Measuring the first three by hand on CPU:")
    rows = {"step, noise 0.01 (default)": run(mjw.step, 0.01),
            "step, noise_std 0": run(mjw.step, 0.0),
            "forward, noise 0.01": run(mjw.forward, 0.01)}
    head = f"run (CPU, nworld={NWORLD}, {NSTEP} steps)"
    print(f"\n{head:<36}{'steps/s':>9}{'ncon/world':>11}{'max|q_w - q_0|':>16}{'d.time':>8}")
    for name, r in rows.items():
        print(f"{name:<36}{r['sps']:>9,.0f}{r['ncon']:>11.2f}{r['spread']:>16.4f}"
              f"{r['time']:>8.3f}")
    print("noise 0: all worlds identical ->", rows["step, noise_std 0"]["spread"] == 0.0)
    print("forward: d.time stays", rows["forward, noise 0.01"]["time"], "(no integration)")

## 4. Solver settings with `-o/--override`
`io.override_model` mutates in place and returns `None`; testspeed applies it to the `MjModel` and again to the `mjw.Model`.

In [ ]:
# Example 25.4 - ex4_overrides.py
"""Example 25.4 - compare solver settings with testspeed's -o/--override.

GPU: four testspeed runs on the humanoid: the defaults, -o "opt.solver=cg",
-o "opt.iterations=5" and -o "opt.cone=elliptic"; prints a comparison table. testspeed's default
--overflow_behavior=error exits 1 on ANY set bit of warn_overflow, and the solver sets the
ITERATIONS bit whenever a world hits the cap, so an unconverged setting would abort the run.
Every row therefore adds -o "opt.warn_overflow=~ITERATIONS|~LS_ITERATIONS" (the mask is read
after the overrides): buffer overflows (NEFC, NARROWPHASE, ...) still abort, and the overflow
column says whether the p95 iteration count sits at the cap. A row that aborts prints
testspeed's "Simulation aborted" lines instead of numbers.
CPU: the same comparison by hand. For each setting we do what cli.load_model/init_structs do:
apply io.override_model to the MjModel, put_model, apply it again to the mjw.Model (it mutates
in place and returns None), then time NSTEP eager steps. CPU numbers are CPU numbers.
Run: python ex4_overrides.py
"""
import subprocess
import sys
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath
from mujoco_warp._src.io import override_model     # private API

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 500 if ON_GPU else 50
MODEL = "test_data/humanoid/humanoid.xml"
SETTINGS = {"defaults": [], "cg": ["opt.solver=cg"], "iterations=5": ["opt.iterations=5"],
            "elliptic": ["opt.cone=elliptic"]}

if ON_GPU:
    import json
    MASK = ["-o", "opt.warn_overflow=~ITERATIONS|~LS_ITERATIONS"]   # iteration caps: report only
    path = str(epath.resource_path("mujoco_warp") / MODEL)
    print(f"{'setting':<16}{'iters':>6}{'steps/s':>14}{'niter mean':>12}{'nefc p95':>10}  overflow")
    for name, ov in SETTINGS.items():
        mjm = mujoco.MjModel.from_xml_path(path)
        override_model(mjm, ov)                         # only to read the iteration cap
        flags = [f for o in ov for f in ("-o", o)] + MASK
        cmd = [sys.executable, "-m", "mujoco_warp.testspeed", MODEL, "--nworld", str(NWORLD),
               "--nstep", str(NSTEP), "--format", "json", *flags]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:                           # e.g. "Simulation aborted: overflow ..."
            lines = [ln for ln in r.stdout.splitlines() if ln.strip()] or r.stderr.splitlines()[-1:]
            print(f"{name:<16}{mjm.opt.iterations:>6}  exited {r.returncode}: " + " / ".join(lines[:3]))
            continue
        m = json.loads(r.stdout.strip().splitlines()[-1])
        cap = "ITERATIONS (p95 at cap)" if m["solver_niter_p95"] >= mjm.opt.iterations else "none"
        print(f"{name:<16}{mjm.opt.iterations:>6}{m['steps_per_second']:>14,.0f}"
              f"{m['solver_niter_mean']:>12.2f}{m['nefc_p95']:>10.0f}  {cap}")
else:
    print("skipped on CPU: mjwarp-testspeed is GPU only; applying the same overrides by hand")
    path = str(epath.resource_path("mujoco_warp") / MODEL)
    print(f"{'setting':<16}{'solver':>8}{'cone':>11}{'iters':>6}{'steps/s':>9}{'niter':>7}"
          f"{'nefc':>6}  overflow")
    for name, ov in SETTINGS.items():
        mjm = mujoco.MjModel.from_xml_path(path)
        if ov:
            override_model(mjm, ov)                     # MjModel side: what C MuJoCo compiles
        mjd = mujoco.MjData(mjm)
        mujoco.mj_resetDataKeyframe(mjm, mjd, 0)
        m = mjw.put_model(mjm)
        if ov:
            assert override_model(m, ov) is None        # mjw.Model side, in place
        m.opt.warn_overflow = 0                         # read the bits instead of printing
        d = mjw.put_data(mjm, mjd, nworld=NWORLD, nconmax=24, njmax=64)
        mjw.step(m, d)                                  # warm-up (JIT) outside the timing
        wp.synchronize()
        niter = []
        t0 = time.perf_counter()
        for _ in range(NSTEP):
            mjw.step(m, d)
            niter.append(d.solver_niter.numpy().mean())  # a host read per step, like testspeed
        wp.synchronize()
        sps = NWORLD * NSTEP / (time.perf_counter() - t0)
        bits = int(np.bitwise_or.reduce(d.overflow.numpy()))
        flags = [o.name for o in mjw.OverflowType
                 if o.value & bits and o.value & (o.value - 1) == 0]
        print(f"{name:<16}{mjw.SolverType(m.opt.solver).name:>8}{mjw.ConeType(m.opt.cone).name:>11}"
              f"{m.opt.iterations:>6}{sps:>9,.0f}{np.mean(niter):>7.2f}{d.nefc.numpy().max():>6}  "
              f"{'none' if not flags else '|'.join(flags)}")

## 5. Control trajectories
`find_keys`, `make_trajectory` (linear interpolation of ctrl keyframes), `load_trajectory` (zero-order hold of an NPZ). Note testspeed's ctrl filter: set `--noise_rate 1e-6` as well as `--noise_std 0` for an exact replay.

In [ ]:
# Example 25.5 - ex5_trajectory_replay.py
"""Example 25.5 - control trajectories: find_keys, make_trajectory, NPZ, load_trajectory, replay.

1. An inline pendulum has three ctrl keyframes named traj_*. io.find_keys picks them by prefix;
   io.make_trajectory linearly interpolates their ctrl on the model timestep.
2. A coarse "log" (one ctrl every 0.03 s) saved as NPZ (ctrl, times, qpos) is read back by
   io.load_trajectory, which resamples it on the timestep with a zero-order hold and sets the
   initial qpos.
3. We replay the trajectory in NWORLD worlds, then show what testspeed's ctrl filter does to it:
   with --replay, every step pulls d.ctrl toward the trajectory at rate exp(-dt/noise_rate), so
   --noise_std 0 alone still lags; the franka_threading benchmark also sets noise_rate 1e-6.
GPU: also runs `mjwarp-testspeed <model> --replay file.npz --noise_std 0 --noise_rate 1e-6`.
Run: python ex5_trajectory_replay.py [--dump-json PATH]
"""
import argparse
import json
import os
import subprocess
import sys
import tempfile

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import io as mjw_io            # private API: find_keys, make/load_trajectory

np.set_printoptions(precision=4, suppress=True)
parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4

PENDULUM = """
<mujoco model="traj_pendulum">
  <option timestep="0.01"/>
  <worldbody>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0" damping="0.1"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" ctrlrange="-5 5" ctrllimited="true"/></actuator>
  <keyframe>
    <key name="home" time="0" ctrl="0"/>
    <key name="traj_0" time="0" ctrl="0"/>
    <key name="traj_1" time="0.05" ctrl="4"/>
    <key name="traj_2" time="0.1" ctrl="-2"/>
  </keyframe>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
dt = mjm.opt.timestep

# 1. keyframes -> linearly interpolated ctrl trajectory
keys = mjw_io.find_keys(mjm, "traj_")
traj = mjw_io.make_trajectory(mjm, keys)
print("find_keys('traj_') ->", keys, "(key 0 'home' is skipped)")
print("make_trajectory shape:", traj.shape, "ctrl:", traj[:, 0])
try:
    mjw_io.make_trajectory(mjm, [keys[1], keys[0]])
except ValueError as e:
    print("ValueError:", e)

# 2. a coarse log -> NPZ -> zero-order hold on the model timestep
log_ctrl = np.array([[0.0], [3.0], [-1.0], [2.0]])
log_times = np.array([0.0, 0.03, 0.06, 0.09])
mjd = mujoco.MjData(mjm)
with tempfile.TemporaryDirectory() as tmp:  # the NPZ is deleted when the block ends
    path = os.path.join(tmp, "ch25_log.npz")
    np.savez(path, ctrl=log_ctrl, times=log_times, qpos=np.array([[0.3]]))
    ctrls = mjw_io.load_trajectory(path, mjm, mjd)
print("load_trajectory shape:", ctrls.shape, "ctrl:", ctrls[:, 0])
print("initial qpos from the NPZ:", mjd.qpos)

# 3. replay in NWORLD worlds: one assign per step (ctrl is live data)
m = mjw.put_model(mjm)
d = mjw.put_data(mjm, mjd, nworld=NWORLD)
for c in ctrls:
    d.ctrl.assign(np.tile(c, (NWORLD, 1)))
    mjw.step(m, d)
q = d.qpos.numpy()[:, 0]
print(f"after {len(ctrls)} steps, qpos per world:", q, "identical:", bool(np.all(q == q[0])))


def testspeed_ctrl(target, noise_rate, lo=-5.0, hi=5.0):
    """The ctrl that cli._ctrl_noise feeds the step for noise_std=0 (start from ctrl = 0)."""
    rate = np.exp(-dt / noise_rate)
    out, c = [], 0.0
    for t in target:
        c = float(np.clip(rate * c + (1 - rate) * t, lo, hi))
        out.append(c)
    return np.array(out)


for nr in (0.1, 1e-6):
    applied = testspeed_ctrl(traj[:, 0], nr)
    print(f"noise_std 0, noise_rate {nr:g}: applied ctrl", applied.round(3),
          f"max lag {np.abs(applied - traj[:, 0]).max():.3f}")

if ON_GPU:
    with tempfile.TemporaryDirectory() as tmp:  # NPZ and XML are deleted after the subprocess
        gpath = os.path.join(tmp, "ch25_traj.npz")
        np.savez(gpath, ctrl=traj, times=np.arange(len(traj)) * dt)
        xml = os.path.join(tmp, "ch25_pendulum.xml")
        with open(xml, "w") as f:
            f.write(PENDULUM)
        cmd = [sys.executable, "-m", "mujoco_warp.testspeed", xml, "--replay", gpath,
               "--noise_std", "0", "--noise_rate", "1e-6", "--nworld", str(NWORLD)]
        print(subprocess.run(cmd, capture_output=True, text=True).stdout[-400:])
else:
    print("skipped on CPU: mjwarp-testspeed --replay (GPU only)")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"dt": dt, "target": traj[:, 0].round(3).tolist(),
                   "lag_0.1": testspeed_ctrl(traj[:, 0], 0.1).round(3).tolist(),
                   "lag_1e-6": testspeed_ctrl(traj[:, 0], 1e-6).round(3).tolist(),
                   "log_times": log_times.tolist(), "log_ctrl": log_ctrl[:, 0].tolist(),
                   "zoh": ctrls[:, 0].round(3).tolist()}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. Size buffers from measured counts
`--measure_alloc` reports per-bucket mean/std/min/max of `nacon` (all worlds) and `nefc` (worst world). Pick p95 + headroom, rerun, and check that nothing overflows.

In [ ]:
# Example 25.6 - ex6_measure_alloc.py
"""Example 25.6 - size nconmax/njmax from measured contact and constraint counts.

GPU: `mjwarp-testspeed ... --measure_alloc` prints per-bucket mean/std/min/max of nacon (all
worlds) and nefc (worst world). We run it on the humanoid, parse the two tables, apply
pick_budget to the bucket max column (testspeed prints no per-step values, so this is an upper
bound on the p95 rule), then rerun testspeed --format json with the default sizes and with the
picked --nconmax/--njmax and compare steps/s and data_memory.
CPU: the same measurement by hand on box_stack(5): a generous first run records, every step,
max(nacon, ncollision) over all worlds and the worst world's nefc (what --measure_alloc
records), prints the bucket tables, chooses budgets with pick_budget (p95 + headroom, rounded
up), and reruns with the default njmax, the measured budget and the generous one.
Run: python ex6_measure_alloc.py [--dump-json PATH]
"""
import argparse
import json
import subprocess
import sys
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 100
BUCKETS = 10 if ON_GPU else 5


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


def pick_budget(maxima, headroom=1.25, multiple=8):
    """Per-world budget: p95 of the measured per-step maxima, times headroom, rounded up."""
    p95 = float(np.percentile(maxima, 95))
    return int(np.ceil(p95 * headroom / multiple) * multiple)


def table(rows, title):
    """testspeed._print_table: one row per bucket of steps."""
    print(f"\n{title}:\n{'mean':>8}{'std':>8}{'min':>6}{'max':>6}")
    for r in rows:
        print(f"{r[0]:>8.1f}{r[1]:>8.2f}{r[2]:>6.0f}{r[3]:>6.0f}")


def buckets(values, n):
    return [[np.mean(c), np.std(c), np.min(c), np.max(c)] for c in np.array_split(values, n)]


if ON_GPU:
    MODEL = "test_data/humanoid/humanoid.xml"
    MASK = ["-o", "opt.warn_overflow=~ITERATIONS|~LS_ITERATIONS"]   # abort on buffers only

    def testspeed(*flags):
        cmd = [sys.executable, "-m", "mujoco_warp.testspeed", MODEL, "--nworld", str(NWORLD),
               "--nstep", str(NSTEP), *MASK, *flags]
        r = subprocess.run(cmd, capture_output=True, text=True)
        return r.returncode, r.stdout

    def bucket_rows(out, title):
        """[mean, std, min, max] rows of one testspeed._print_table block."""
        lines = out.splitlines()
        i = lines.index(f"{title}:") + 4      # skip title, blank line, header, separator
        return np.array([[float(x) for x in ln.split()] for ln in lines[i:i + BUCKETS]])

    code, out = testspeed("--measure_alloc", "--num_buckets", str(BUCKETS))
    if code != 0:
        raise RuntimeError("testspeed --measure_alloc failed:\n" + out[-2000:])
    print(out[out.index("nacon alloc:"):])
    nacon, nefc = bucket_rows(out, "nacon alloc"), bucket_rows(out, "nefc alloc")
    nconmax = pick_budget(nacon[:, 3] / NWORLD)           # bucket max: an upper bound
    njmax = pick_budget(nefc[:, 3])
    print(f"bucket max per world: ncon {nacon[:, 3].max() / NWORLD:.1f}, nefc {nefc[:, 3].max():.0f}"
          f" -> pick_budget: nconmax={nconmax}, njmax={njmax}")

    print(f"\n{'budget':<16}{'steps/s':>14}{'data MiB':>10}  result")
    for label, flags in [("default sizes", []),
                         ("measured", ["--nconmax", str(nconmax), "--njmax", str(njmax)])]:
        code, out = testspeed("--format", "json", *flags)
        if code != 0:                                     # "Simulation aborted: overflow ..."
            print(f"{label:<16}  exited {code}: " + " / ".join(out.strip().splitlines()[-3:]))
            continue
        m = json.loads(out.strip().splitlines()[-1])
        print(f"{label:<16}{m['steps_per_second']:>14,.0f}{m['data_memory'] / 2**20:>10.1f}  "
              f"{m['converged_worlds']}/{NWORLD} converged, no buffer overflow")
else:
    print("skipped on CPU: --measure_alloc is a testspeed flag; measuring box_stack(5) by hand")
    mjm = mujoco.MjModel.from_xml_string(box_stack(5))
    rng = np.random.default_rng(0)
    q0 = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
    for b in range(5):                                    # small xy jitter per world and box
        q0[:, 7 * b:7 * b + 2] += rng.uniform(-0.03, 0.03, (NWORLD, 2)).astype(np.float32)


    def run(nconmax, njmax, record=False):
        m = mjw.put_model(mjm)
        m.opt.warn_overflow = 0
        d = mjw.make_data(mjm, nworld=NWORLD, nconmax=nconmax, njmax=njmax)
        d.qpos.assign(q0)
        mjw.step(m, d)                                    # warm-up
        d.qpos.assign(q0)
        d.qvel.zero_()
        wp.synchronize()
        nacon, nefc, t = [], [], 0.0
        for _ in range(NSTEP):
            t0 = time.perf_counter()
            mjw.step(m, d)
            wp.synchronize()
            t += time.perf_counter() - t0
            if record:
                nacon.append(max(d.nacon.numpy()[0], d.ncollision.numpy()[0]))
                nefc.append(d.nefc.numpy().max())
        bits = int(np.bitwise_or.reduce(d.overflow.numpy()))
        top_z = float(d.qpos.numpy()[:, 7 * 4 + 2].mean())
        return np.array(nacon), np.array(nefc), NWORLD * NSTEP / t, bits, top_z, d


    nacon, nefc, *_ , d0 = run(nconmax=64, njmax=256, record=True)
    print(f"generous run: nworld={NWORLD}, naconmax={d0.naconmax}, njmax={d0.njmax}, {NSTEP} steps")
    table(buckets(nacon, BUCKETS), "nacon alloc (all worlds)")
    table(buckets(nefc, BUCKETS), "nefc alloc (worst world)")

    ncon_world = nacon / NWORLD
    nconmax = pick_budget(ncon_world)
    njmax = pick_budget(nefc)
    print(f"\np95 per world: ncon {np.percentile(ncon_world, 95):.1f}, "
          f"nefc {np.percentile(nefc, 95):.1f}"
          f" -> pick_budget: nconmax={nconmax}, njmax={njmax}")

    names = {int(o): o.name for o in mjw.OverflowType if int(o) & (int(o) - 1) == 0 and int(o)}
    print(f"\n{'budget':<22}{'nconmax':>8}{'njmax':>6}{'steps/s':>9}{'top box z':>10}  overflow")
    results = []
    for label, nc, nj in [("default njmax", None, None), ("measured (p95+25%)", nconmax, njmax),
                          ("generous", 64, 256)]:
        _, _, sps, bits, z, d = run(nc, nj)
        flags = "|".join(n for v, n in names.items() if bits & v) or "none"
        print(f"{label:<22}{d.naconmax // NWORLD:>8}{d.njmax:>6}{sps:>9,.0f}{z:>10.3f}  {flags}")
        results.append({"label": label, "nconmax": d.naconmax // NWORLD, "njmax": d.njmax,
                        "sps": round(sps), "z": round(z, 3), "overflow": flags})
    print("stable stack: top box near z = 0.9 (0.1 + 4 * 0.2, minus soft-contact penetration)")

    if args.dump_json:
        with open(args.dump_json, "w") as f:
            json.dump({"nworld": NWORLD, "nstep": NSTEP,
                       "nacon": (nacon / NWORLD).round(2).tolist(), "nefc": nefc.tolist(),
                       "nconmax": nconmax, "njmax": njmax, "runs": results},
                      f, separators=(",", ":"))
        print("wrote", args.dump_json)

## 7. Memory versus nworld
Data grows linearly with nworld; `njmax` sets the slope of the constraint arrays. On a GPU the example also prints the nworld that fits in memory.

In [ ]:
# Example 25.7 - ex7_memory_vs_nworld.py
"""Example 25.7 - memory of mjw.Data versus nworld, and which fields dominate.

We add up the capacity of every wp.array in Model and Data (what testspeed's --memory reports
as "Model memory" and "Data memory"), for the humanoid with nconmax=24, njmax=64. Data grows
linearly with nworld, so we measure small batches, check the line, and extrapolate the table
for 1k..64k worlds. A second line shows how njmax changes the slope.
GPU only: the largest nworld that fits in wp.get_device().total_memory (on CPU that value
is host RAM when psutil is installed, and 0 with a warning otherwise, so we skip it).
Run: python ex7_memory_vs_nworld.py [--dump-json PATH]
"""
import argparse
import dataclasses
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
MEASURE = [1, 64, 256, 1024] if not ON_GPU else [1, 1024, 8192]


def field_bytes(obj, prefix=""):
    """{field: bytes} for every wp.array in a (nested) dataclass, like testspeed --memory."""
    out = {}
    for f in dataclasses.fields(obj):
        v = getattr(obj, f.name)
        if dataclasses.is_dataclass(v):
            out.update(field_bytes(v, prefix + f.name + "."))
        elif isinstance(v, wp.array):
            out[prefix + f.name] = v.capacity
    return out


def group(name):
    if name.startswith("efc."):
        return "efc (constraints)"
    if name.startswith("contact."):
        return "contact pool"
    if name in ("qM", "M", "qLD", "qLDiagInv", "qH", "qHLD", "qLU", "qLDiagSqrtInv"):
        return "mass/Hessian factors"
    return "everything else"


mjm = mujoco.MjModel.from_xml_path(str(epath.resource_path("mujoco_warp")
                                       / "test_data/humanoid/humanoid.xml"))
model_mb = sum(field_bytes(mjw.put_model(mjm)).values()) / 2**20
print(f"Model memory: {model_mb:.3f} MiB (shared by all worlds)")

sizes = {}
for n in MEASURE:
    d = mjw.make_data(mjm, nworld=n, nconmax=24, njmax=64)
    sizes[n] = field_bytes(d)
    del d
per_world = (sum(sizes[MEASURE[-1]].values()) - sum(sizes[1].values())) / (MEASURE[-1] - 1)
fixed = sum(sizes[1].values()) - per_world
for n in MEASURE:
    pred = fixed + per_world * n
    print(f"  nworld {n:>5}: measured {sum(sizes[n].values()) / 2**20:9.3f} MiB, line "
          f"{pred / 2**20:9.3f} MiB")
print(f"Data = {fixed:.0f} B + {per_world:,.1f} B x nworld (nconmax=24, njmax=64)")

groups = {}
for k, v in sizes[MEASURE[-1]].items():
    groups[group(k)] = groups.get(group(k), 0) + v / MEASURE[-1]
print("per-world bytes by group:",
      {g: round(b) for g, b in sorted(groups.items(), key=lambda x: -x[1])})

d256 = field_bytes(mjw.make_data(mjm, nworld=64, nconmax=24, njmax=256))
pw256 = sum(d256.values()) / 64
print(f"njmax=256 instead of 64: {pw256:,.0f} B per world ({pw256 / per_world:.2f}x)")

print(f"\n{'nworld':>7}{'Data MiB (njmax 64)':>22}{'Data MiB (njmax 256)':>23}")
table = []
for n in [1024, 4096, 8192, 16384, 32768, 65536]:
    a, b = (fixed + per_world * n) / 2**20, pw256 * n / 2**20
    table.append([n, round(a, 1), round(b, 1)])
    print(f"{n:>7}{a:>22.1f}{b:>23.1f}")

if ON_GPU:
    total = wp.get_device().total_memory
    print(f"device total_memory: {total / 2**30:.1f} GiB")
    print(f"max nworld in device memory (Data only, njmax 64): {int(total // per_world):,}")
else:   # on CPU total_memory is host RAM (via psutil; 0 plus a warning without it)
    print("skipped on CPU: no device-memory estimate")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"per_world": round(per_world, 1), "per_world_njmax256": round(pw256, 1),
                   "groups": {g: round(b) for g, b in groups.items()},
                   "groups_njmax256": {
                       g: round(sum(v for k, v in d256.items() if group(k) == g) / 64)
                       for g in groups},
                   "table": table}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 8. A tuning study on `box_stack(8)`
One knob per row; a row only counts if it has no overflow and the correctness probe (top box height) does not move. Graph replay on GPU.

In [ ]:
# Example 25.8 - ex8_tuning_study.py
"""Example 25.8 - apply the tuning checklist to box_stack(8), one knob at a time.

Each row changes one thing relative to the row it builds on and reports world-steps/s, the mean
solver iteration count, the overflow flags and a correctness probe (mean height of the top box
after NSTEP steps; a settled 8-box stack keeps it near 1.5). A faster row that overflows or moves
the probe is not a valid speed-up. Eager stepping on CPU; CUDA-graph replay on GPU (Ch 24).
CPU numbers are CPU numbers: they rank Python + serial kernels, not GPU throughput.
Run: python ex8_tuning_study.py [--dump-json PATH]
"""
import argparse
import json
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.io import override_model     # private API

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 60
NBOX = 8


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


mjm = mujoco.MjModel.from_xml_string(box_stack(NBOX))
m0 = mjw.put_model(mjm)
print(f"box_stack({NBOX}): nv={mjm.nv}, opt.iterations={mjm.opt.iterations}, "
      f"ls_iterations={mjm.opt.ls_iterations}, broadphase={m0.opt.broadphase.name}")
print("launch config m.block_dim (experimental): update_gradient_grad =",
      m0.block_dim.update_gradient_grad, "(nv rounded up to 32, clamped to 32..256),",
      "nxn_broadphase =", m0.block_dim.nxn_broadphase, "| on CPU Warp runs block_dim 1")

# label, sizes (nconmax, njmax), overrides; each row builds on an earlier one
SIZED = (40, 160)
ROWS = [
    ("1 defaults", (None, None), []),
    ("2 + sized nconmax/njmax", SIZED, []),
    ("3 + iterations=10, ls_iterations=10", SIZED, ["opt.iterations=10", "opt.ls_iterations=10"]),
    ("4 + broadphase=sap_tile", SIZED, ["opt.iterations=10", "opt.ls_iterations=10",
                                         "opt.broadphase=sap_tile"]),
    ("5 cost of deterministic=true", SIZED, ["opt.iterations=10", "opt.ls_iterations=10",
                                              "opt.deterministic=true"]),
    ("6 too far: cg, iterations=3", SIZED, ["opt.solver=cg", "opt.iterations=3"]),
]
NAMES = {int(o): o.name for o in mjw.OverflowType if int(o) and int(o) & (int(o) - 1) == 0}


def run(sizes, overrides):
    m = mjw.put_model(mjm)
    if overrides:
        override_model(m, overrides)
    m.opt.warn_overflow = 0                         # wp.static: set before the first launch
    if ON_GPU and not wp.is_conditional_graph_supported():
        m.opt.graph_conditional = False
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=sizes[0], njmax=sizes[1])
    mjw.step(m, d)                                  # warm-up / JIT
    mjw.reset_data(m, d)
    wp.synchronize()
    if ON_GPU:
        with wp.ScopedCapture() as cap:
            mjw.step(m, d)
        mjw.reset_data(m, d)
        step = lambda: wp.capture_launch(cap.graph)          # noqa: E731
    else:
        step = lambda: mjw.step(m, d)                        # noqa: E731
    t0 = time.perf_counter()
    for _ in range(NSTEP):
        step()
    wp.synchronize()
    sps = NWORLD * NSTEP / (time.perf_counter() - t0)
    bits = int(np.bitwise_or.reduce(d.overflow.numpy()))
    flags = "|".join(n for v, n in NAMES.items() if bits & v) or "none"
    top = float(d.qpos.numpy()[:, 7 * (NBOX - 1) + 2].mean())
    return sps, float(d.solver_niter.numpy().mean()), flags, top, d


print(f"\n{'row (CPU eager)' if not ON_GPU else 'row (GPU graph)':<38}{'naconmax':>9}{'njmax':>6}"
      f"{'steps/s':>9}{'niter':>6}{'top z':>7}  overflow")
results = []
for label, sizes, ov in ROWS:
    sps, nit, flags, top, d = run(sizes, ov)
    print(f"{label:<38}{d.naconmax:>9}{d.njmax:>6}{sps:>9,.0f}{nit:>6.1f}{top:>7.3f}  {flags}")
    results.append({"row": label, "naconmax": d.naconmax, "njmax": d.njmax, "sps": round(sps),
                    "niter": round(nit, 2), "top_z": round(top, 3), "overflow": flags})

ok = [r for r in results
      if r["overflow"] == "none" and abs(r["top_z"] - results[1]["top_z"]) < 0.01]
print("\nvalid rows (no overflow, probe unchanged):", ", ".join(r["row"][0] for r in ok))
print("invalid rows:", ", ".join(f"{r['row'][0]} ({r['overflow']}, top z {r['top_z']})"
                                 for r in results if r not in ok))
if ON_GPU:
    best = max(ok, key=lambda r: r["sps"])
    print(f"fastest valid row: {best['row']} ({best['sps'] / results[1]['sps']:.2f}x row 2)")
else:
    sps = [r["sps"] for r in ok]
    print(f"CPU: valid rows differ by {100 * (max(sps) / min(sps) - 1):.0f}% "
          "(timing noise at this size); rank them on a GPU")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "nstep": NSTEP, "device": str(wp.get_device()),
                   "rows": results}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## Part VI tuning checklist
Print it, then fill in your own GPU numbers next to each item.

In [ ]:
# Part VI tuning checklist (Ch 25 section 8). Fill in your own numbers on the GPU.
CHECKLIST = [
    ("nworld",            "raise until world-steps/s stops growing (Ch 24 sweep)"),
    ("nconmax / njmax",   "p95 of --measure_alloc + headroom; zero overflow (Ch 9, Example 25.6)"),
    ("solver + iterations", "Newton for stiff contact; cap iterations near the converged count (Ch 20)"),
    ("cone",              "pyramidal: 2(condim-1) rows per contact; elliptic: condim rows; measure"),
    ("broadphase",        "NXN for small scenes, SAP_TILE / SAP_SEGMENTED for many geoms (Ch 15)"),
    ("integrator",        "Euler / implicitfast; RK4 runs forward 4 times per step (Ch 13)"),
    ("determinism",       "off unless you need bitwise replays (Ch 22)"),
    ("sleep",             "enable for scenes that come to rest (Ch 21)"),
    ("CUDA graphs",       "capture step (+ your kernels); recapture after baked changes (Ch 24)"),
    ("warn_overflow",     "set before the first launch: it is wp.static (recompile)"),
]
for knob, rule in CHECKLIST:
    print(f"[ ] {knob:<20} {rule}")

## Exercise 25.1: `pick_budget(maxima, headroom)`
Turn the sizing rule into a function and assert it on the data from Example 25.6.

In [ ]:
import numpy as np

# Per-step maxima measured by Example 25.6 on box_stack(5), nworld=8, 100 steps (CPU run):
# contacts per world (max(nacon, ncollision) / nworld) and constraint rows of the worst world.
NCON = [1, 4, 4, 8, 8, 12, 12, 16] + [20] * 92
NEFC = [0, 16, 16, 32, 32, 48, 48, 64] + [80] * 92


def pick_budget(maxima, headroom=1.25, multiple=8):
    """Return the smallest multiple of `multiple` that is >= p95(maxima) * headroom."""
    ...


assert pick_budget(NCON) == 32
assert pick_budget(NEFC) == 104
assert pick_budget(NEFC, headroom=1.0) == 80
assert pick_budget([10] * 20, headroom=1.0) == 16
print("pick_budget OK:", pick_budget(NCON), pick_budget(NEFC))

In [ ]:
#@title Exercise 25.1: solution (double-click to show the code)

import numpy as np

# Per-step maxima measured by Example 25.6 on box_stack(5), nworld=8, 100 steps (CPU run):
# contacts per world (max(nacon, ncollision) / nworld) and constraint rows of the worst world.
NCON = [1, 4, 4, 8, 8, 12, 12, 16] + [20] * 92
NEFC = [0, 16, 16, 32, 32, 48, 48, 64] + [80] * 92


def pick_budget(maxima, headroom=1.25, multiple=8):
    """Return the smallest multiple of `multiple` that is >= p95(maxima) * headroom."""
    p95 = float(np.percentile(maxima, 95))
    return int(np.ceil(p95 * headroom / multiple) * multiple)


assert pick_budget(NCON) == 32
assert pick_budget(NEFC) == 104
assert pick_budget(NEFC, headroom=1.0) == 80
assert pick_budget([10] * 20, headroom=1.0) == 16
print("pick_budget OK:", pick_budget(NCON), pick_budget(NEFC))

## Exercise 25.2: `parse_testspeed(json_text, nworld)`
Parse one `--format json` line: throughput, ns per world-step, the converged check and the trace children of a prefix.

In [ ]:
import json

# A subset of one testspeed --format json line, built from a single run: the CPU stand-in trace of
# Example 25.1 (nworld=8, 20 steps, data/ex1_trace.json), trace keys in ns per world-step with the
# 4.0.1 tree, and steps_per_second = 1e9 / step so both agree. CPU numbers, not GPU performance.
SAMPLE = (
    '{"run_time": 0.2015, "steps_per_second": 794.2167, "converged_worlds": 8, '
    '"step": 1259102.15, "step.forward": 1219653.79, "step.euler": 38863.05, '
    '"step.forward.fwd_position": 691028.73, '
    '"step.forward.fwd_position.fwd_kinematics": 81632.62, '
    '"step.forward.fwd_position.collision": 452981.55, '
    '"step.forward.fwd_position.fwd_kinematics.kinematics": 34824.56}'
)


def parse_testspeed(json_text, nworld):
    """Return {"sps", "ns_per_world_step", "ok", "children"} from one testspeed JSON line.

    ok: every world converged; children(prefix) -> {child name: ns} one level below prefix."""
    ...


r = parse_testspeed(SAMPLE, nworld=8)
assert round(r["sps"]) == 794
assert abs(r["ns_per_world_step"] - 1e9 / 794.2167) < 1e-6
assert r["ok"] is True and parse_testspeed(SAMPLE, nworld=16)["ok"] is False
assert r["children"]("step") == {"forward": 1219653.79, "euler": 38863.05}
assert r["children"]("step.forward.fwd_position") == {"fwd_kinematics": 81632.62,
                                                      "collision": 452981.55}
assert r["children"]("step.forward.fwd_position.fwd_kinematics") == {"kinematics": 34824.56}
print("parse_testspeed OK")

In [ ]:
#@title Exercise 25.2: solution (double-click to show the code)

import json

# A subset of one testspeed --format json line, built from a single run: the CPU stand-in trace of
# Example 25.1 (nworld=8, 20 steps, data/ex1_trace.json), trace keys in ns per world-step with the
# 4.0.1 tree, and steps_per_second = 1e9 / step so both agree. CPU numbers, not GPU performance.
SAMPLE = (
    '{"run_time": 0.2015, "steps_per_second": 794.2167, "converged_worlds": 8, '
    '"step": 1259102.15, "step.forward": 1219653.79, "step.euler": 38863.05, '
    '"step.forward.fwd_position": 691028.73, '
    '"step.forward.fwd_position.fwd_kinematics": 81632.62, '
    '"step.forward.fwd_position.collision": 452981.55, '
    '"step.forward.fwd_position.fwd_kinematics.kinematics": 34824.56}'
)


def parse_testspeed(json_text, nworld):
    """Return {"sps", "ns_per_world_step", "ok", "children"} from one testspeed JSON line.

    ok: every world converged; children(prefix) -> {child name: ns} one level below prefix."""
    m = json.loads(json_text.strip().splitlines()[-1])
    sps = m["steps_per_second"]

    def children(prefix):
        depth = prefix.count(".") + 1
        return {k[len(prefix) + 1:]: v for k, v in m.items()
                if k.startswith(prefix + ".") and k.count(".") == depth}

    return {"sps": sps, "ns_per_world_step": 1e9 / sps, "ok": m["converged_worlds"] == nworld,
            "children": children}


r = parse_testspeed(SAMPLE, nworld=8)
assert round(r["sps"]) == 794
assert abs(r["ns_per_world_step"] - 1e9 / 794.2167) < 1e-6
assert r["ok"] is True and parse_testspeed(SAMPLE, nworld=16)["ok"] is False
assert r["children"]("step") == {"forward": 1219653.79, "euler": 38863.05}
assert r["children"]("step.forward.fwd_position") == {"fwd_kinematics": 81632.62,
                                                      "collision": 452981.55}
assert r["children"]("step.forward.fwd_position.fwd_kinematics") == {"kinematics": 34824.56}
print("parse_testspeed OK")

## Stretch: reproduce a nightly benchmark row on your GPU

In [ ]:
# Stretch (GPU): reproduce the nightly humanoid row on your Colab GPU.
# The humanoid benchmark is {"nworld": 8192, "nconmax": 24, "njmax": 64} (benchmarks/humanoid/
# __init__.py) and run.py adds --event_trace --memory --measure_solver --measure_alloc.
# The shipped test_data humanoid is a close stand-in; for the exact MJCF clone the repo:
#   !git clone -q https://github.com/google-deepmind/mujoco_warp && cd mujoco_warp && git checkout 8f5b9f1
#   !mjwarp-testspeed mujoco_warp/benchmarks/humanoid/humanoid.xml --nworld 8192 --nconmax 24 \
#        --njmax 64 --format json --event_trace
# Compare steps_per_second and step.forward.* with your own trace from Example 25.1.
if ON_GPU:
    !mjwarp-testspeed test_data/humanoid/humanoid.xml --nworld 8192 --nconmax 24 --njmax 64 --format short --event_trace
else:
    print("skipped on CPU: mjwarp-testspeed is GPU only")